# freeCAM PI-atm

This notebook follows the same object-oriented UI as FreeCESM. Machine paths, run-directory preparation, PBS submission, `mpiexec`, and the persistent socket are implementation details hidden by `freecam.Driver`.

In [ ]:
%load_ext autoreload
%autoreload 2

import freecam as fc
import numpy as np

## 1. Create one model

Constructing the driver is cheap and does not submit a job. The first live-state operation lazily prepares a private run directory, submits one cpudev job when needed, and starts the persistent 512-rank CAM session.

In [ ]:
driver = fc.Driver(case='PI-atm', nsteps=2)
print(driver.case)
print('50-step validation:', driver.validation.get('bfb', 'see validation record'))

In [ ]:
fig, axes = driver.cam.state.plot(rank=0, label='initial')
print(driver.cam.state.summary(rank=0))
print('persistent job:', driver.cam.status.get('job_id', 'managed by Driver'))
print('run directory:', driver.run_dir)

## 2. Inspect and run the Python-owned workflow

The workflow is a live view of the current phase/process order. `execute()` runs complete CAM steps and returns the actual worker-side action trace.

In [ ]:
driver.cam.workflow

In [ ]:
trace = driver.execute(verbose=False)
print(f'executed {len(trace)} actions across {driver.nsteps} complete steps')
print('first:', trace[0]['phase'], trace[0]['name'])
print('last: ', trace[-1]['phase'], trace[-1]['name'])

for axis, variable in zip(axes.flat, ('T', 'u', 'v', 'q')):
    driver.cam.state.plot_profile(
        variable, rank=0, ax=axis, color='tab:orange', label='after 2 steps'
    )
print(driver.cam.state.summary(rank=0))

## 3. Add a variable and a Python physics process

Attribute assignment is declarative: every MPI rank resolves the named dimensions against its local grid, allocates a Fortran-contiguous NumPy array, and registers it in that rank's StatePool. A `Physics` object can then be inserted directly into the live workflow.

In [ ]:
driver.cam.state.experiment_tracer = fc.Variable(
    dims=('pcols', 'pver', 'chunks'),
    units='kg kg-1',
    initial=0.0,
    aliases=('tracer',),
    standard_name='experiment_tracer',
)

class NotebookTracer(fc.Physics):
    name = 'notebook_tracer_source'
    phase = 'cam_run1'
    after = 'dadadj'
    writes = ('tracer',)

    def tendency(self, fields, context):
        fields['tracer'][...] += 1.0e-6 * context.timestep_seconds

tracer_process = driver.cam.workflow.insert(NotebookTracer())
driver.cam.workflow

In [ ]:
# Run only this process; the model clock does not advance.
tracer_process.run()
print(driver.cam.state.experiment_tracer.stats(rank='global'))

# The same object controls its placement and on/off state.
tracer_process.move(before='deep_convection')
tracer_process.disable()
tracer_process.enable()

## 4. Load an original Fortran process at runtime

The same state syntax defines its inputs. freeCAM generates the `bind(C)` adapter, compiles the device `.so`, loads it on every rank, and inserts the process into the workflow.

In [ ]:
driver.cam.state.runtime_temperature = fc.Variable(
    dims=('nphys_local', 'pver'),
    units='K',
    initial=240.0,
    standard_name='runtime_plugin_temperature',
)
driver.cam.state.runtime_temperature_increment = fc.Variable(
    dims=(),
    units='K',
    initial=1.5,
    writable=False,
    standard_name='runtime_plugin_temperature_increment',
)

fortran_process = driver.cam.physics.install_fortran(
    driver.repo / 'examples/plugins/runtime_temperature_offset/device.yaml',
    project_root=driver.repo,
    process='runtime_temperature_offset',
    phase='cam_run1',
    after='dadadj',
    unsafe=True,
)
fortran_process.run()
driver.cam.state.runtime_temperature.stats(rank='global')

## 5. Fine-grained experiments

A single process or phase can be run directly for controlled experiments. These calls do not automatically run prerequisites or advance model time.

In [ ]:
dadadj_trace = driver.cam.physics.dadadj.run()
cam_run1_trace = driver.cam.phases.cam_run1.run()
print('dadadj:', dadadj_trace)
print('cam_run1 actions:', len(cam_run1_trace))

## 6. Remove runtime extensions and close the model

Processes must be removed before deleting fields they use. `driver.close()` finalizes CAM and releases the persistent PBS/MPI session.

In [ ]:
tracer_process.remove()
fortran_process.remove()

del driver.cam.state.experiment_tracer
del driver.cam.state.runtime_temperature
del driver.cam.state.runtime_temperature_increment

driver.close()
print('closed:', not driver.running)